# Human fibroblast/decidual pseudotime, re-rooted at the outer fibroblasts (eFib1)

The pseudotime of 7 Nov 2026 (`pseudotime_human_final_1107.ipynb`) was rooted by the rule "earliest-phase fibroblast at
the fibroblast end of the trajectory diffusion component", which picked an eFib2 nucleus. On the diffusion map eFib1
(outer fibroblasts, the human counterpart of the mouse root subtype) lies beyond eFib2 at the fibroblast end and also
carries a program of its own in later components, so diffusion pseudotime from an eFib2 root gave eFib1 a mid value
(0.43) and the order eFib2 < eFib3 < eFib1 < eFib4 < dec1 < dec2 (test of 12 Nov 2026,
`$NH/tmp/2026_11_12_reroot_efib1/`).

This notebook keeps everything of the 7 Nov run (scVI latent, moscot couplings, CellRank kernel, the 15-component
diffusion map and its eigenvalues, all stored in the 7 Nov object) and changes only the root: the same rule, restricted
to the outer-fibroblast subtype (`ROOT_LABEL`), as in mouse where the root is an uninduced eFib1 nucleus. DPT is
recomputed with scanpy exactly as before (`sc.tl.dpt`, 15 components) and rank-normalized. Before the new root is used,
the notebook checks that DPT from the 7 Nov root reproduces the stored pseudotime exactly.

The output is a copy of the input object with `obs['pseudotime']` replaced; the 7 Nov pseudotime is kept in
`obs['pseudotime_root_eFib2_20261107']`, and `uns['pseudotime']` records both roots. The input is only read.


In [ ]:
# Parameters (papermill can override these)
INPUT_OBJECT = "/n/lab_storage/eddy_lab/Lab/mckinley/data/20261107_Human_fibdec_final_nick_pdt.h5ad"
OUTPUT_OBJECT = ("/n/netscratch/eddy_lab/Everyone/nhilgert/2026_11_12_human_reroot/"
                 "20261112_Human_fibdec_final_nick_pdt.h5ad")
ROOT_LABEL = "eFib1"   # root: the earliest-phase nucleus of this subtype at the fibroblast end of the trajectory DC


## Setup

In [ ]:
import datetime
import hashlib
import os
import re
import time
import warnings
from importlib.metadata import version as pkg_version

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
versions = {p: pkg_version(p) for p in ("scanpy", "anndata", "numpy", "scipy")}
print(versions)

LABEL_COL = "cell_subtype"
# Fibroblast -> decidual order; it orients the trajectory component
ORDER = ["eFib1", "eFib2", "eFib3", "eFib4", "dec1", "dec2"]
is_fibroblast = lambda labels: labels.str.startswith("eFib")
DAY_OF = {"secretory": 0.0, "menstrual": 1.0}


def day_of(timepoint):
    return DAY_OF.get(re.sub(r"[\s_\-]", "", str(timepoint).lower()))


def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 24), b""):
            digest.update(block)
    return digest.hexdigest()


## The 7 Nov object: labels, stored diffusion map and eigenvalues

In [ ]:
t_start = time.time()
obj = sc.read_h5ad(INPUT_OBJECT)
input_sha256 = sha256_of(INPUT_OBJECT)
print(f"{INPUT_OBJECT}: {obj.n_obs:,} nuclei x {obj.n_vars:,} genes (sha256 {input_sha256[:12]}...)")
record = {k: (v.decode() if isinstance(v, bytes) else v) for k, v in obj.uns["pseudotime"].items()}
P = {k: (float(v) if "." in v or "e" in v else int(v)) if v.replace(".", "").replace("-", "").isdigit() else v
     for k, v in record["params"].items()}
evals = np.array([float(x) for x in re.findall(r"[-\d.eE+]+", str(record["diffmap_evals"]))])
n_dcs = int(record["n_dcs"])
assert evals.size == n_dcs == obj.obsm["X_diffmap"].shape[1]
print(f"7 Nov run: root {record['root']} ({record['root_label']}), trajectory DC{record['trajectory_dc']}, "
      f"{n_dcs} components, {int(record['n_cells']):,} nuclei; params {record['params']}")

labels_all = obj.obs[LABEL_COL].astype(str)
in_run = np.isfinite(obj.obsm["X_diffmap"]).all(axis=1) & obj.obs["pseudotime"].notna().to_numpy()
assert in_run.sum() == int(record["n_cells"])
a = ad.AnnData(obs=obj.obs.loc[in_run, [LABEL_COL, "timepoint", "sample"]].astype(str))
a.obsm["X_diffmap"] = np.asarray(obj.obsm["X_diffmap"][in_run], dtype=float)
a.obsm["X_scVI"] = np.asarray(obj.obsm["X_scVI_pseudotime"][in_run], dtype=np.float32)
a.uns["diffmap_evals"] = evals
a.obs["day"] = a.obs.timepoint.map(day_of).astype(float)
assert a.obs.day.notna().all()
DM = a.obsm["X_diffmap"]
lab = a.obs[LABEL_COL]
print(f"{a.n_obs:,} nuclei in the run; {int((~in_run).sum()):,} without a pseudotime (apoptotic decidual nuclei)")
print(lab.value_counts().reindex(ORDER).to_string())


## Check: DPT from the 7 Nov root reproduces the stored pseudotime

`sc.tl.dpt` needs a neighbors entry (not used by the distance itself); it is built on the stored latent as in the
7 Nov notebook.

In [ ]:
sc.pp.neighbors(a, use_rep="X_scVI", n_neighbors=int(P["n_neighbors"]), random_state=int(P["seed"]))
record_root = int(a.obs_names.get_loc(record["root"]))


def rank_normalized_dpt(root):
    a.uns["iroot"] = int(root)
    sc.tl.dpt(a, n_dcs=n_dcs)
    pt = a.obs["dpt_pseudotime"].to_numpy(float)
    assert np.isfinite(pt).all() and pt[root] == 0
    return pd.Series(pt).rank(pct=True).to_numpy()


pt_check = rank_normalized_dpt(record_root)
stored = obj.obs.loc[in_run, "pseudotime"].to_numpy(float)
# The 7 Nov object stores the eigenvalues as a printed string (8 significant digits), so the DPT weights
# lambda / (1 - lambda) are rounded and a few near-tied nuclei can swap rank; allow 3 ranks of 44,949.
max_rank_diff = np.abs(pt_check - stored).max() * a.n_obs
print(f"max |DPT from the 7 Nov root - stored pseudotime| = {np.abs(pt_check - stored).max():.2e} "
      f"({max_rank_diff:.1f} ranks of {a.n_obs:,}); Spearman {spearmanr(pt_check, stored).statistic:.6f}")
assert max_rank_diff <= 3, "the stored pseudotime is not reproduced; stop"


## Root: the 7 Nov rule, restricted to the outer-fibroblast subtype

The trajectory component and its fibroblast end are found as in the 7 Nov notebook (the component whose rank
correlation with the fibroblast -> decidual label order is strongest). The root is the earliest-phase nucleus of
`ROOT_LABEL` at that end.

In [ ]:
order_rank = lab.map({s: i for i, s in enumerate(ORDER)}).to_numpy(float)
assert np.isfinite(order_rank).all(), sorted(set(lab[~np.isfinite(order_rank)]))
rhos = {i: float(spearmanr(DM[:, i], order_rank).statistic) for i in range(1, 6)}
best = max(rhos, key=lambda i: abs(rhos[i]))
fib_end = "max" if rhos[best] < 0 else "min"
assert best == int(record["trajectory_dc"]), (best, record["trajectory_dc"])
earliest = a.obs.day == a.obs.day.min()

# The 7 Nov rule (any fibroblast) and the same rule restricted to ROOT_LABEL
pool_record = (is_fibroblast(lab) & earliest).to_numpy()
pool_label = ((lab == ROOT_LABEL) & earliest).to_numpy()
pick = lambda pool: int(np.nanargmax(np.where(pool, DM[:, best], np.nan)) if fib_end == "max"
                        else np.nanargmin(np.where(pool, DM[:, best], np.nan)))
root_record_rule, root = pick(pool_record), pick(pool_label)
assert a.obs_names[root_record_rule] == record["root"], "the 7 Nov rule no longer gives the 7 Nov root"
print(f"trajectory component DC{best} (rho {rhos[best]:+.3f}), fibroblast end = {fib_end}")
print(f"7 Nov rule, any fibroblast: {a.obs_names[root_record_rule]} ({lab.iloc[root_record_rule]})")
print(f"same rule, {ROOT_LABEL} only:    {a.obs_names[root]} ({lab.iloc[root]}, {a.obs.timepoint.iloc[root]})")
dc1 = DM[:, best] if fib_end == "min" else -DM[:, best]
print("mean position of each subtype on the trajectory component (fibroblast end first):")
print(pd.Series(dc1).groupby(lab.to_numpy()).mean().reindex(ORDER).round(4).to_string())


## DPT from the new root, rank-normalized

In [ ]:
pt_rank = rank_normalized_dpt(root)
dv = a.obs.day.to_numpy(float)
rho_day = float(spearmanr(pt_rank, dv).statistic)
rho_old = float(spearmanr(pt_rank, stored).statistic)
medians = pd.Series(pt_rank).groupby(lab.to_numpy()).median().reindex(ORDER)
print(f"pseudotime on {a.n_obs:,} nuclei; root {a.obs_names[root]}; Spearman rho with phase {rho_day:.2f}; "
      f"with the 7 Nov pseudotime {rho_old:.3f}")
print("median pseudotime by subtype:", " < ".join(f"{s} {v:.2f}" for s, v in medians.sort_values().items()))
assert medians[ROOT_LABEL] == medians.min(), "the root subtype is not the earliest"


## Diffusion map with both roots

In [ ]:
colors = dict(zip(ORDER, ["#08306b", "#2171b5", "#9ecae1", "#41ab5d", "#fdae6b", "#e6550d"]))
rng = np.random.default_rng(0)
shuffle = rng.permutation(a.n_obs)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (i, j) in zip(axes, [(1, 2), (1, 4)]):
    ax.scatter(DM[shuffle, i], DM[shuffle, j], s=0.4, c=[colors[s] for s in lab.iloc[shuffle]], rasterized=True)
    ax.scatter(DM[record_root, i], DM[record_root, j], s=120, marker="*", c="k", label="7 Nov root (eFib2)")
    ax.scatter(DM[root, i], DM[root, j], s=80, marker="X", c="magenta", label=f"new root ({ROOT_LABEL})")
    ax.set_xlabel(f"DC{i}"); ax.set_ylabel(f"DC{j}"); ax.set_xticks([]); ax.set_yticks([])
for s in ORDER:
    axes[0].scatter([], [], c=colors[s], label=s, s=20)
axes[0].legend(frameon=False, fontsize=8, markerscale=1.5)
fig.suptitle(f"Diffusion map of {a.n_obs:,} nuclei (7 Nov run); roots marked", fontsize=10)
fig.tight_layout()
plt.show()


## Pseudotime by subtype and phase, old and new

In [ ]:
both = pd.DataFrame({"subtype": lab.to_numpy(), "phase": a.obs.timepoint.to_numpy(),
                     "7 Nov (eFib2 root)": stored, f"new ({ROOT_LABEL} root)": pt_rank})
table = (both.groupby(["subtype", "phase"]).median().round(3).unstack("phase").reindex(ORDER))
print(table.to_string())
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, col in zip(axes, ["7 Nov (eFib2 root)", f"new ({ROOT_LABEL} root)"]):
    ax.violinplot([both.loc[both.subtype == s, col] for s in ORDER], showmedians=True)
    ax.set_xticks(range(1, len(ORDER) + 1), ORDER); ax.set_title(col); ax.set_ylabel("pseudotime (rank / N)")
fig.tight_layout()
plt.show()


## Output object

In [ ]:
out = obj.copy()
pseudotime = pd.Series(np.nan, index=out.obs_names)
pseudotime[a.obs_names] = pt_rank
out.obs["pseudotime_root_eFib2_20261107"] = obj.obs["pseudotime"].to_numpy(float)
out.obs["pseudotime"] = pseudotime.to_numpy()
settings = dict(record)
settings.update(
    root=str(a.obs_names[root]), root_label=str(lab.iloc[root]),
    root_rule=f"earliest-phase {ROOT_LABEL} nucleus at the fibroblast end of the trajectory component",
    root_record_20261107=record["root"], root_label_record_20261107=record["root_label"],
    rerooted=(f"{datetime.datetime.now():%Y-%m-%d}: DPT recomputed from the {ROOT_LABEL} root on the diffusion map of "
              "the 7 Nov run (pseudotime_human_reroot_1112.ipynb); the 7 Nov pseudotime is kept in "
              "obs['pseudotime_root_eFib2_20261107']"),
    rho_with_timepoint=rho_day, rho_with_pseudotime_20261107=rho_old,
    reproduction_of_20261107_max_rank_diff=float(max_rank_diff),
    input_object=INPUT_OBJECT, input_sha256=input_sha256, versions_reroot=versions,
    computed=datetime.datetime.now().isoformat(timespec="seconds"))
out.uns["pseudotime"] = settings
os.makedirs(os.path.dirname(os.path.abspath(OUTPUT_OBJECT)), exist_ok=True)
assert not os.path.exists(OUTPUT_OBJECT), f"{OUTPUT_OBJECT} exists; choose a new OUTPUT_OBJECT"
out.write_h5ad(OUTPUT_OBJECT, compression="gzip")
print(f"wrote {OUTPUT_OBJECT} ({out.n_obs:,} nuclei; {int(out.obs['pseudotime'].notna().sum()):,} with pseudotime) "
      f"in {time.time() - t_start:.0f} s; sha256 {sha256_of(OUTPUT_OBJECT)[:12]}...")
